# NeuroLIF-SKY130 — Weighted Synapse Characterization

This notebook documents the corrected 2-bit electronic synapse.

The final implementation deliberately uses **identical PMOS unit
current cells** rather than a geometrically doubled PMOS device.

Final encoding:

- `00` → 0 active unit cells
- `01` → 1 active unit cell
- `10` → 2 identical unit cells in parallel
- `11` → 3 active unit cells


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root():
    start = Path.cwd().resolve()

    candidates = [
        start,
        *start.parents,
    ]

    for candidate in candidates:

        if (
            (candidate / "results").exists()
            and
            (candidate / "layout").exists()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate NeuroLIF-SKY130 repository root."
    )


ROOT = find_repo_root()

print("Repository root:", ROOT)


In [ ]:
RESULT_FILE = (
    ROOT
    / "results"
    / "synapse"
    / "weight_current_summary.csv"
)


synapse_df = pd.read_csv(
    RESULT_FILE,
    dtype={
        "weight": str,
    },
)


synapse_df["weight"] = (
    synapse_df["weight"]
    .str.zfill(2)
)


synapse_df


## Unit-cell weighting result

During development, a nominally 2×-wide PMOS did **not** deliver
2× the unit-cell current in the selected SKY130 configuration.

The observed characterization was approximately:

- one unit PMOS: **103.738 nA**
- one nominally 2×-wide PMOS: **98.196 nA**
- two identical unit PMOS devices in parallel: **207.476 nA**

Therefore the final synapse uses replicated identical unit cells.
No stronger claim is made about the exact physical cause of the
wide-device behavior.


In [ ]:
table = (
    synapse_df
    .set_index("weight")
)


i01 = float(
    table.loc[
        "01",
        "median_delivered_current_nA",
    ]
)

i10 = float(
    table.loc[
        "10",
        "median_delivered_current_nA",
    ]
)

i11 = float(
    table.loc[
        "11",
        "median_delivered_current_nA",
    ]
)


ratio_2x = i10 / i01
ratio_3x = i11 / i01


print(
    f"I01 = {i01:.4f} nA"
)

print(
    f"I10 = {i10:.4f} nA"
)

print(
    f"I11 = {i11:.4f} nA"
)

print()

print(
    f"I10 / I01 = {ratio_2x:.6f}"
)

print(
    f"I11 / I01 = {ratio_3x:.6f}"
)


In [ ]:
fig, ax = plt.subplots(
    figsize=(7, 4.5)
)

ax.bar(
    synapse_df["weight"],
    synapse_df[
        "median_delivered_current_nA"
    ],
)

ax.set_xlabel(
    "2-bit weight"
)

ax.set_ylabel(
    "Median delivered current (nA)"
)

ax.set_title(
    "Final unit-cell weighted synapse"
)

ax.grid(
    axis="y",
    alpha=0.25,
)

plt.tight_layout()
plt.show()


In [ ]:
WEIGHT_SCALING_PASS = bool(

    1.95 <= ratio_2x <= 2.05

    and

    2.95 <= ratio_3x <= 3.05
)


print(
    "WEIGHT_SCALING_PASS =",
    WEIGHT_SCALING_PASS,
)


assert WEIGHT_SCALING_PASS


## Final implementation decision

The final layout therefore contains three identical PMOS current
cells:

- one cell assigned to the 1× bit
- two identical cells assigned to the 2× bit

When both bits are active, all three unit cells contribute.

This is an implementation and characterization result rather than
a claim of a novel current-mirror principle.
